# 26_Full_Pipeline_Summary_Audit
## Materia Arche Agent OS v3.0 — Final Audit

This notebook consolidates the complete Materia Arche pipeline from NB01 through NB25. No new computation — this is a reference document that links every result, correction, and decision to its source notebook.

In [1]:
import pandas as pd
import numpy as np
import yaml
import os
import warnings
warnings.filterwarnings('ignore')

# Load lock files
with open('contracts/schema.lock.yaml') as f:
    schema = yaml.safe_load(f)
with open('contracts/model.lock.yaml') as f:
    model_lock = yaml.safe_load(f)
with open('contracts/benchmark.lock.yaml') as f:
    benchmark = yaml.safe_load(f)

print("Materia Arche — Full Pipeline Summary Audit")
print(f"Lock files loaded: schema v{schema['version']}, {model_lock['model_id']}, benchmark {benchmark['benchmark_id']}")

Materia Arche — Full Pipeline Summary Audit
Lock files loaded: schema vv1, ET_baseline_v1, benchmark kendall_taub_cv


## 1. Dataset

In [2]:
df = pd.read_csv('perovskite_stability_clean.csv')

print("=" * 70)
print("DATASET SUMMARY")
print("=" * 70)
print(f"  Source: Perovskite Database Project (Zenodo 16809654)")
print(f"  Full database: 41,916 devices")
print(f"  With T80 stability data: {len(df)} devices")
print(f"  Prepared in: NB01")
print(f"  Target: Stability_PCE_T80 (transformed via log1p)")
print(f"  Features: {len(schema['columns'])}")
for col in schema['columns']:
    print(f"    - {col}")
print(f"")
print(f"  Target statistics:")
t80 = df['Stability_PCE_T80']
print(f"    Median: {t80.median():.0f}h")
print(f"    Mean: {t80.mean():.0f}h")
print(f"    Range: [{t80.min():.0f}, {t80.max():.0f}]h")
print(f"    Devices >1000h: {(t80 > 1000).sum()} ({(t80 > 1000).mean():.1%})")

DATASET SUMMARY
  Source: Perovskite Database Project (Zenodo 16809654)
  Full database: 41,916 devices
  With T80 stability data: 1543 devices
  Prepared in: NB01
  Target: Stability_PCE_T80 (transformed via log1p)
  Features: 16
    - Perovskite_band_gap
    - Pb
    - Sn
    - I
    - Br
    - Cl
    - MA
    - FA
    - Cs
    - first_Prvskt_annealing_temperature
    - first_Prvskt_thermal_annealing_time
    - Perovskite_thickness
    - Cell_area_measured
    - JV_default_Voc
    - JV_default_Jsc
    - JV_default_FF

  Target statistics:
    Median: 150h
    Mean: 374h
    Range: [0, 8400]h
    Devices >1000h: 109 (7.1%)


## 2. Model evolution

In [3]:
print("=" * 70)
print("MODEL EVOLUTION")
print("=" * 70)
print("""
  NB02: Random Forest baseline
    tau-b: 0.249 (seed=42, 200 trees)
    First benchmark established

  NB13: Model sweep (4 algorithms × 200+ configs)
    ExtraTrees identified as winner
    XGBoost/LightGBM excluded (no OpenMP runtime)

  NB14: Feature engineering attempt
    8 engineered features tested → hurt performance (-0.016)
    Lesson: trees already learn interactions

  NB15: Final grid search (96 configs × 5-fold CV)
    LOCKED MODEL: ExtraTreesRegressor
      n_estimators: 700
      max_features: sqrt
      min_samples_split: 3
      min_samples_leaf: 1
      bootstrap: False
    Test tau-b: 0.2714 (p < 10⁻¹⁰)
    CV tau-b: 0.2889 (5-fold)
    Bootstrap: ET better in 81.8% of 1000 resamples
    BUT: 95% CI for lift includes zero
""")

MODEL EVOLUTION

  NB02: Random Forest baseline
    tau-b: 0.249 (seed=42, 200 trees)
    First benchmark established

  NB13: Model sweep (4 algorithms × 200+ configs)
    ExtraTrees identified as winner
    XGBoost/LightGBM excluded (no OpenMP runtime)

  NB14: Feature engineering attempt
    8 engineered features tested → hurt performance (-0.016)
    Lesson: trees already learn interactions

  NB15: Final grid search (96 configs × 5-fold CV)
    LOCKED MODEL: ExtraTreesRegressor
      n_estimators: 700
      max_features: sqrt
      min_samples_split: 3
      min_samples_leaf: 1
      bootstrap: False
    Test tau-b: 0.2714 (p < 10⁻¹⁰)
    CV tau-b: 0.2889 (5-fold)
    Bootstrap: ET better in 81.8% of 1000 resamples
    BUT: 95% CI for lift includes zero



## 3. Quantum track

In [4]:
print("=" * 70)
print("QUANTUM EXPERIMENTS — CLOSED")
print("=" * 70)
print("""
  NB08: 6 quantum feature encoding experiments → 0/6 positive lift
  NB09: Trained variational circuit → no lift
  NB10: Per-device VQE energy → no lift (9th experiment)
  NB11: Decision gate — quantum track closed

  Conclusion: Quantum composition encoding does not improve stability
  prediction. This is an honest negative result. Real quantum chemistry
  (DFT+VQE) remains a separate research direction, not a gate.

  Status: Negative (reusable asset: methodology documented)
""")

QUANTUM EXPERIMENTS — CLOSED

  NB08: 6 quantum feature encoding experiments → 0/6 positive lift
  NB09: Trained variational circuit → no lift
  NB10: Per-device VQE energy → no lift (9th experiment)
  NB11: Decision gate — quantum track closed

  Conclusion: Quantum composition encoding does not improve stability
  prediction. This is an honest negative result. Real quantum chemistry
  (DFT+VQE) remains a separate research direction, not a gate.

  Status: Negative (reusable asset: methodology documented)



## 4. Agent OS work packets (P-001 through P-008)

In [5]:
packets = [
    {'id': 'P-001', 'nb': 18, 'title': 'Reproducibility audit',
     'question': 'Is the locked ET model robust across splits?',
     'result': 'Mean tau-b 0.303 ± 0.042 across 20 splits',
     'status': 'Promising', 'decision': 'Iterate',
     'note': 'Mean good but std 0.042 > 0.03 threshold. Footer corrected (was inflated).'},
    {'id': 'P-002', 'nb': 19, 'title': 'Variance reduction',
     'question': 'Can we reduce tau-b variance below 0.03?',
     'result': '4 strategies tested, best std 0.036 (top-8 SHAP features)',
     'status': 'Negative', 'decision': 'Stop',
     'note': 'Variance is structural for 1,543 devices. Not tunable.'},
    {'id': 'P-003', 'nb': 20, 'title': 'Uncertainty ranking',
     'question': 'Can we add meaningful uncertainty to rankings?',
     'result': '1/10 top candidates separated from median. Conservative ranking drops tau-b to 0.114.',
     'status': 'Promising', 'decision': 'Iterate',
     'note': 'Intervals are context, not filters. Mean ranking is best.'},
    {'id': 'P-004', 'nb': 21, 'title': 'Shortlist robustness',
     'question': 'Is the NB17 lab panel stable across splits?',
     'result': '0/3 frozen candidates consistently in top-20',
     'status': 'Negative', 'decision': 'Stop',
     'note': 'Full-dataset ranking was biased (train memorization). Corrected in P-005.'},
    {'id': 'P-005', 'nb': 22, 'title': 'Robust lab panel',
     'question': 'Can we produce a split-robust shortlist?',
     'result': '31 candidates qualify. Top 3: devices 850, 1374, 848 (all 100% top-20 rate)',
     'status': 'Confirmed', 'decision': 'Advance',
     'note': 'Test-set-only ranking. First Confirmed packet.'},
    {'id': 'P-006', 'nb': 23, 'title': 'Panel profiling',
     'question': 'Are candidates compositionally diverse?',
     'result': 'All 3 are MA-only. 4 families available in qualified pool.',
     'status': 'Promising', 'decision': 'Iterate',
     'note': 'Diversity gap identified. Diversified panel recommended.'},
    {'id': 'P-007', 'nb': 24, 'title': 'Diversified panel lock',
     'question': 'Can we lock a diverse panel for Phase 2?',
     'result': 'Devices 850, 1064, 119 — 3 families, all 100% top-20 rate',
     'status': 'Confirmed', 'decision': 'Advance',
     'note': 'E4 validation-ready. Panel locked.'},
    {'id': 'P-008', 'nb': 25, 'title': 'Phase 2 outreach package',
     'question': 'Do we have a complete partner-ready package?',
     'result': '9/9 components present. ISOS-L-1 protocol, 2-tier budget, success criteria.',
     'status': 'Confirmed', 'decision': 'Advance',
     'note': 'Release ceiling: Partner Draft.'},
]

print("=" * 70)
print("AGENT OS WORK PACKETS — COMPLETE RECORD")
print("=" * 70)

for p in packets:
    marker = "***" if p['status'] == 'Confirmed' else "   "
    print(f"\n{marker} {p['id']} (NB{p['nb']}): {p['title']}")
    print(f"    Question: {p['question']}")
    print(f"    Result: {p['result']}")
    print(f"    Status: {p['status']} / {p['decision']}")
    print(f"    Note: {p['note']}")

confirmed = sum(1 for p in packets if p['status'] == 'Confirmed')
negative = sum(1 for p in packets if p['status'] == 'Negative')
promising = sum(1 for p in packets if p['status'] == 'Promising')
print(f"\n  Summary: {confirmed} Confirmed, {promising} Promising, {negative} Negative out of {len(packets)} packets")

AGENT OS WORK PACKETS — COMPLETE RECORD

    P-001 (NB18): Reproducibility audit
    Question: Is the locked ET model robust across splits?
    Result: Mean tau-b 0.303 ± 0.042 across 20 splits
    Status: Promising / Iterate
    Note: Mean good but std 0.042 > 0.03 threshold. Footer corrected (was inflated).

    P-002 (NB19): Variance reduction
    Question: Can we reduce tau-b variance below 0.03?
    Result: 4 strategies tested, best std 0.036 (top-8 SHAP features)
    Status: Negative / Stop
    Note: Variance is structural for 1,543 devices. Not tunable.

    P-003 (NB20): Uncertainty ranking
    Question: Can we add meaningful uncertainty to rankings?
    Result: 1/10 top candidates separated from median. Conservative ranking drops tau-b to 0.114.
    Status: Promising / Iterate
    Note: Intervals are context, not filters. Mean ranking is best.

    P-004 (NB21): Shortlist robustness
    Question: Is the NB17 lab panel stable across splits?
    Result: 0/3 frozen candidates con

## 5. Corrections log

In [6]:
print("=" * 70)
print("CORRECTIONS LOG")
print("=" * 70)
print("""
  1. NB01-NB13: Early notebooks contained overstated claims and
     non-reproducible results. See README_Honesty_Note.md for details.
     From NB14 onward, all results are real, local, and reproducible.

  2. NB18 (P-001): Footer stated "Model is confirmed robust for Phase 2
     candidates" despite status being Promising/Iterate (std 0.042 > 0.03).
     Violated Law #7: No claim above earned evidence level.
     Corrected in NB19 correction notice and NB18 re-executed.

  3. NB21 (P-004): Used full-dataset rankings including training data,
     causing memorization bias. Corrected in NB22 (P-005) with
     test-set-only evaluation methodology.

  4. NB22 (P-005): Original panel all MA-only — lacked compositional
     diversity. Caught by P-006, corrected in P-007 with diversified panel.
""")

CORRECTIONS LOG

  1. NB01-NB13: Early notebooks contained overstated claims and
     non-reproducible results. See README_Honesty_Note.md for details.
     From NB14 onward, all results are real, local, and reproducible.

  2. NB18 (P-001): Footer stated "Model is confirmed robust for Phase 2
     candidates" despite status being Promising/Iterate (std 0.042 > 0.03).
     Violated Law #7: No claim above earned evidence level.
     Corrected in NB19 correction notice and NB18 re-executed.

  3. NB21 (P-004): Used full-dataset rankings including training data,
     causing memorization bias. Corrected in NB22 (P-005) with
     test-set-only evaluation methodology.

  4. NB22 (P-005): Original panel all MA-only — lacked compositional
     diversity. Caught by P-006, corrected in P-007 with diversified panel.



## 6. Locked panel

In [7]:
print("=" * 70)
print("LOCKED DIVERSIFIED PANEL (P-007)")
print("=" * 70)
print("""
  #1  Device 850  — MA₃Pb₄I₁₃ (MA-only)
      T80: 3400h | Prediction: 3118h (80% CI: 2735–3400h)
      Top-20 rate: 100% (4/4) | Mean rank: 1.0 ± 0.0
      Rationale: Most rank-stable device in dataset

  #2  Device 1064 — FA₀.₈₅MA₀.₁₅PbI₃ (MA-FA mixed)
      T80: 5400h | Prediction: 5027h (80% CI: 5400–6735h)
      Top-20 rate: 100% (5/5) | Mean rank: 3.4 ± 3.0
      Rationale: Highest T80 in qualified pool, FA-rich diversity

  #3  Device 119  — FA₀.₈₃Cs₀.₁₇PbI₂Br (FA-Cs triple)
      T80: 3423h | Prediction: 1886h (80% CI: 1499–3423h)
      Top-20 rate: 100% (4/4) | Mean rank: 9.5 ± 1.8
      Rationale: Triple-cation mixed halide, tests Cs stabilization

  Selection methodology:
    1. 20-split test-set-only ranking (P-005)
    2. ≥80% top-20 appearance rate + T80 ≥500h → 31 qualified
    3. Composition diversity check (P-006) → all MA-only, failed
    4. Diversified from 4 available families (P-007) → 3 families, all 100%
""")

LOCKED DIVERSIFIED PANEL (P-007)

  #1  Device 850  — MA₃Pb₄I₁₃ (MA-only)
      T80: 3400h | Prediction: 3118h (80% CI: 2735–3400h)
      Top-20 rate: 100% (4/4) | Mean rank: 1.0 ± 0.0
      Rationale: Most rank-stable device in dataset

  #2  Device 1064 — FA₀.₈₅MA₀.₁₅PbI₃ (MA-FA mixed)
      T80: 5400h | Prediction: 5027h (80% CI: 5400–6735h)
      Top-20 rate: 100% (5/5) | Mean rank: 3.4 ± 3.0
      Rationale: Highest T80 in qualified pool, FA-rich diversity

  #3  Device 119  — FA₀.₈₃Cs₀.₁₇PbI₂Br (FA-Cs triple)
      T80: 3423h | Prediction: 1886h (80% CI: 1499–3423h)
      Top-20 rate: 100% (4/4) | Mean rank: 9.5 ± 1.8
      Rationale: Triple-cation mixed halide, tests Cs stabilization

  Selection methodology:
    1. 20-split test-set-only ranking (P-005)
    2. ≥80% top-20 appearance rate + T80 ≥500h → 31 qualified
    3. Composition diversity check (P-006) → all MA-only, failed
    4. Diversified from 4 available families (P-007) → 3 families, all 100%



## 7. Full notebook index

In [8]:
notebooks = [
    (1, 'Data preparation', 'Load and clean Perovskite Database, extract T80 stability'),
    (2, 'ML baseline', 'Random Forest baseline (tau-b 0.249)'),
    (3, 'Candidate shortlist', 'Rank compositions, identify candidates'),
    (4, 'Ablation & evidence', 'Feature importance, ablation study'),
    (5, 'Phase 2 preparation', 'Lab scoping, decision framework'),
    (6, 'Outreach package v1', 'Draft outreach templates'),
    (7, 'Lab quotes & evidence', 'Lab cost estimates, evidence packaging'),
    (8, 'Quantum diagnostics', 'Diagnose quantum feature failure (6 experiments)'),
    (9, 'Quantum re-test', 'Trained variational circuit, milestone re-eval'),
    (10, 'VQE experiment', 'Per-device VQE energy, 9th experiment fails'),
    (11, 'Decision gate', 'Close quantum track, present Options A/B/C'),
    (12, 'Phase 2 outreach', 'Final outreach templates, lab scoping table'),
    (13, 'ML variation sweep', '4 models × 30 configs — ExtraTrees wins'),
    (14, 'Deepen winner', 'Feature engineering, ensemble tests'),
    (15, 'Final ML push', 'Grid search, bootstrap CIs, production model locked'),
    (16, 'Nitrogen roadmap', 'What nitrogen fixation ML actually needs'),
    (17, 'SHAP + lab panel', 'SHAP analysis, 3-candidate selection for Phase 2'),
    (18, 'P-001 Reproducibility', '20-split reproducibility audit of locked ET model'),
    (19, 'P-002 Variance reduction', '4 strategies tested — variance is structural'),
    (20, 'P-003 Uncertainty ranking', 'Per-tree prediction intervals, 1/10 separated'),
    (21, 'P-004 Shortlist robustness', 'Frozen top-3 not stable across splits'),
    (22, 'P-005 Robust lab panel', 'New 3-candidate panel via test-set-only ranking'),
    (23, 'P-006 Panel profiling', 'Composition diversity check — all MA-only'),
    (24, 'P-007 Diversified panel lock', '3 families, all 100% top-20 — E4 validation-ready'),
    (25, 'P-008 Phase 2 outreach', 'Complete partner-ready package with protocol and budget'),
    (26, 'Pipeline summary audit', 'This notebook — consolidates all results'),
]

print("=" * 70)
print("FULL NOTEBOOK INDEX")
print("=" * 70)
print(f"{'#':>3} {'Title':<30} {'Description'}")
print("-" * 70)
for num, title, desc in notebooks:
    print(f"{num:>3}  {title:<30} {desc}")
print(f"\n  Total: {len(notebooks)} notebooks")

FULL NOTEBOOK INDEX
  # Title                          Description
----------------------------------------------------------------------
  1  Data preparation               Load and clean Perovskite Database, extract T80 stability
  2  ML baseline                    Random Forest baseline (tau-b 0.249)
  3  Candidate shortlist            Rank compositions, identify candidates
  4  Ablation & evidence            Feature importance, ablation study
  5  Phase 2 preparation            Lab scoping, decision framework
  6  Outreach package v1            Draft outreach templates
  7  Lab quotes & evidence          Lab cost estimates, evidence packaging
  8  Quantum diagnostics            Diagnose quantum feature failure (6 experiments)
  9  Quantum re-test                Trained variational circuit, milestone re-eval
 10  VQE experiment                 Per-device VQE energy, 9th experiment fails
 11  Decision gate                  Close quantum track, present Options A/B/C
 12  Phase 2 outre

## 8. Evidence level assessment

In [9]:
print("=" * 70)
print("EVIDENCE LEVEL ASSESSMENT")
print("=" * 70)
print("""
  E0 Idea                    — Complete (NB01)
  E1 Runnable baseline       — Complete (NB02: RF tau-b 0.249)
  E2 Benchmarked comparison  — Complete (NB15: ET tau-b 0.271, CV 0.289)
  E3 Decision-grade shortlist — Complete (P-005/P-007: diversified panel)
  E4 Validation-ready package — Complete (P-008: partner-ready outreach)
  E5 Physical confirmation   — NOT STARTED (requires lab partner)

  Current evidence level: E4
  Next step: Send P-008 outreach package to lab partner
  
  Honest caveats at E4:
    - Model lift over RF is real but modest (95% CI includes zero)
    - Prediction intervals are wide (model ranks better than it predicts)
    - Variance across splits is structural (~0.04), not reducible by tuning
    - 3 compositions / 15 devices is a pilot, not definitive validation
    - Early notebooks (01-13) contain overstated claims (see Honesty Note)
""")

EVIDENCE LEVEL ASSESSMENT

  E0 Idea                    — Complete (NB01)
  E1 Runnable baseline       — Complete (NB02: RF tau-b 0.249)
  E2 Benchmarked comparison  — Complete (NB15: ET tau-b 0.271, CV 0.289)
  E3 Decision-grade shortlist — Complete (P-005/P-007: diversified panel)
  E4 Validation-ready package — Complete (P-008: partner-ready outreach)
  E5 Physical confirmation   — NOT STARTED (requires lab partner)

  Current evidence level: E4
  Next step: Send P-008 outreach package to lab partner
  
  Honest caveats at E4:
    - Model lift over RF is real but modest (95% CI includes zero)
    - Prediction intervals are wide (model ranks better than it predicts)
    - Variance across splits is structural (~0.04), not reducible by tuning
    - 3 compositions / 15 devices is a pilot, not definitive validation
    - Early notebooks (01-13) contain overstated claims (see Honesty Note)



## 9. Honest status footer

In [10]:
print("=" * 65)
print("HONEST STATUS — MATERIA ARCHE v3.0")
print("=" * 65)
print(f"Notebook: 26 (Pipeline Summary Audit)")
print(f"Status: Confirmed")
print(f"Evidence level: E4 (validation-ready package)")
print(f"Decision outcome: Advance to Phase 2 (requires lab partner)")
print(f"Release ceiling: Internal")
print(f"Domain: perovskite")
print(f"Dataset version: perovskite_stability_clean.csv (1,543 devices)")
print(f"")
print(f"Pipeline stats:")
print(f"  26 notebooks")
print(f"  8 work packets (3 Confirmed, 2 Promising, 3 Negative)")
print(f"  4 corrections documented and applied")
print(f"  9 quantum experiments (0 positive — honestly closed)")
print(f"  1 locked model (ExtraTrees, tau-b 0.271)")
print(f"  1 locked diversified panel (3 compositions, 3 families)")
print(f"  1 partner-ready outreach package")
print(f"")
print(f"What changes next: E5 requires physical lab validation")
print(f"")
print(f"Reviewer sign-off: Evidence Guardian __________")

HONEST STATUS — MATERIA ARCHE v3.0
Notebook: 26 (Pipeline Summary Audit)
Status: Confirmed
Evidence level: E4 (validation-ready package)
Decision outcome: Advance to Phase 2 (requires lab partner)
Release ceiling: Internal
Domain: perovskite
Dataset version: perovskite_stability_clean.csv (1,543 devices)

Pipeline stats:
  26 notebooks
  8 work packets (3 Confirmed, 2 Promising, 3 Negative)
  4 corrections documented and applied
  9 quantum experiments (0 positive — honestly closed)
  1 locked model (ExtraTrees, tau-b 0.271)
  1 locked diversified panel (3 compositions, 3 families)
  1 partner-ready outreach package

What changes next: E5 requires physical lab validation

Reviewer sign-off: Evidence Guardian __________
